# Вероятность победы карты: составы и история игроков

Базовая модель до драфта: Elo конкретной пятёрки, средний Elo игроков, сглаженная форма
последних 20 завершённых карт и опыт. При смене пятёрки командная история сбрасывается,
опыт игроков переносится между командами. Каждый пример — одна карта, не две стороны.

Исторические пятёрки берутся из сыгранных матчей. Оценка ниже предполагает **известную
пятёрку до начала карты**; это условный сценарий, а не проверка прогноза по старому составу
до объявления замены. Для будущей карты пятёрки и время передаются явно.

Результат доступен только после `start_time + duration`. Мы не используем результат,
длительность, драфт или статистику самой предсказываемой карты как признаки.
Снимки текущих профилей и их карьерные итоги не используются как исторические признаки.

На демо модель не обучается. Минимум 80 реальных карт — технический порог этого MVP,
не гарантия статистически достаточной выборки. Это не прогноз победителя серии bo3/bo5.


In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from dota_scout.analysis import load_bundle, analysis_frames, to_match
from dota_scout.forecast import valid_lineup

default_path = ROOT / "data/analysis.json"
DATA_PATH = os.getenv("SCOUT_DATA_PATH") or str(default_path if default_path.exists()
    else ROOT / "tests/fixtures/demo.json")
BUNDLE = load_bundle(DATA_PATH)
ANALYSIS_START = pd.Timestamp(os.getenv("SCOUT_ANALYSIS_START", "2026-01-01"), tz="UTC")
print("Источник:", DATA_PATH)
print("Синтетические данные:", bool(BUNDLE.get("synthetic")))
print("Дата экспорта:", BUNDLE.get("exported_at", "демо"))
print("Матчей в исходном наборе:", len(BUNDLE["matches"]))
if not BUNDLE["matches"]:
    raise ValueError("Выборка пуста. Сначала загрузите матчи.")
teams, players, drafts = analysis_frames(BUNDLE)
if teams.empty:
    raise ValueError("Нет матчей с известными team_id.")
report = teams[teams.started_at >= ANALYSIS_START].copy()
print("Период анализа:", ANALYSIS_START, "—", teams.started_at.max())


## Признаки во времени
Более ранние сезоны могут дать историю игрокам, но обзор результатов начинается в 2026 году.


In [ ]:
from dota_scout.forecast import FEATURES, chronological_features, temporal_holdout, history_before, probability
HISTORY_START = pd.Timestamp(os.getenv("SCOUT_HISTORY_START", "2023-01-01"), tz="UTC")
history_matches = [to_match(m) for m in BUNDLE["matches"]
    if m["start_time"] >= HISTORY_START.timestamp() and m.get("leagueid") != 999999]
feature_rows = chronological_features(history_matches)
feature_frame = pd.DataFrame(feature_rows)
print("Карт с пригодными пятёрками:", len(feature_rows))
if not feature_frame.empty:
    display(feature_frame.head())
else:
    print("Нужны реальные матчи с известными ID всех десяти игроков.")


## Хронологическая проверка

Последние 25% времени — тест. Результаты тренировочных карт, не завершённых к границе,
удаляются. Все preprocessing-параметры обучаются только на тренировочной части.
В течение теста рейтинги обновляются уже завершившимися картами, коэффициенты модели
не переобучаются: это последовательная онлайн-проверка.

Сравниваем с вероятностью 50% и простым Elo. Log loss и Brier меньше — лучше;
accuracy сама по себе не проверяет качество вероятностей.


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss
from sklearn.calibration import calibration_curve

model = None
predictions = None
real_count = len(feature_rows)
if BUNDLE.get("synthetic") or real_count < 80:
    print(f"Модель не обучена: {real_count} пригодных реальных карт; нужно минимум 80.")
else:
    try:
        train_rows, test_rows, split_time = temporal_holdout(feature_rows)
    except ValueError as exc:
        print("Проверка невозможна:", exc)
    else:
        train, test = pd.DataFrame(train_rows), pd.DataFrame(test_rows)
        model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0,max_iter=2000,random_state=42))
        model.fit(train[FEATURES],train.radiant_win)
        estimates = {
            "50%":np.full(len(test),0.5),
            "Roster Elo":1/(1+10**(-test.roster_elo_diff.to_numpy()/400)),
            "Logistic":model.predict_proba(test[FEATURES])[:,1],
        }
        scores=[]
        for name,p in estimates.items():
            scores.append({"model":name,"maps":len(test),
                "accuracy":accuracy_score(test.radiant_win,p>=0.5),
                "brier":brier_score_loss(test.radiant_win,p),
                "log_loss":log_loss(test.radiant_win,p,labels=[0,1])})
        display(pd.DataFrame(scores))
        print("Граница теста:",pd.to_datetime(split_time,unit="s",utc=True))
        print("Train / test:",len(train),len(test))
        predictions=test[["match_id","started_at","radiant_win"]].copy()
        predictions["p_radiant"]=estimates["Logistic"]
        display(predictions.tail(15))
        observed,expected=calibration_curve(test.radiant_win,estimates["Logistic"],n_bins=5,strategy="quantile")
        plt.plot(expected,observed,"o-",label="Logistic")
        plt.plot([0,1],[0,1],"--",label="идеальная калибровка")
        plt.xlabel("предсказанная вероятность"); plt.ylabel("доля побед")
        plt.legend(); plt.show()


## Прогноз конкретной будущей карты

Нужно передать точные team_id, десять Steam32 ID, сторону и время прогноза в UTC.
Состав последней сыгранной карты автоматически не считается будущим составом.
Функция переобучается только на картах, завершённых до указанного времени.
Для нового состава может быть ноль совместных игр — это показывается в результате.
Значение 0.60 означает оценку 60%, а не гарантированную победу.


In [ ]:
def predict_future(radiant_team_id,dire_team_id,radiant_five,dire_five,as_of):
    cutoff=pd.Timestamp(as_of)
    if cutoff.tzinfo is None:
        raise ValueError("Укажи timezone, например 2026-10-05T12:00:00Z")
    usable=[r for r in feature_rows if r["finished_at"] < cutoff.timestamp()]
    if BUNDLE.get("synthetic") or len(usable)<80 or len({r["radiant_win"] for r in usable})<2:
        raise ValueError("Недостаточно реальной истории для обучения")
    fit=pd.DataFrame(usable)
    fitted=make_pipeline(StandardScaler(),LogisticRegression(C=1.0,max_iter=2000,random_state=42))
    fitted.fit(fit[FEATURES],fit.radiant_win)
    history=history_before(history_matches,cutoff.timestamp())
    features=history.features(radiant_team_id,dire_team_id,tuple(radiant_five),tuple(dire_five))
    p=float(fitted.predict_proba(pd.DataFrame([features])[FEATURES])[0,1])
    return {"as_of":cutoff.isoformat(),"p_radiant":p,"p_dire":1-p,
            "train_maps":len(usable),"radiant_roster_games":features["radiant_roster_games"],
            "dire_roster_games":features["dire_roster_games"],"scenario":"known supplied five, pre-draft"}

# Заполни известные до начала матча составы; автоматический прогноз не запускается.
# predict_future(radiant_team_id=...,dire_team_id=...,
#     radiant_five=[...],dire_five=[...],as_of="2026-10-05T12:00:00Z")


## Ограничения

Модель пока не учитывает роли, официальный статус стендинов, драфт, составы тренеров,
серию bo3/bo5 и изменение баланса патчами. Игроки с короткой историей получают исходный Elo.
Название турнира не доказывает tier-1; семейства берутся из пользовательского справочника.
После накопления данных нужна проверка по сезонам/патчам и отдельная временная выборка
для настройки и калибровки. Не выбирай модель по результату одного маленького теста.
